# Q4: Machine Translation (English → German)

This notebook compares a recurrent Seq2Seq model with Bahdanau attention against a pretrained Transformer-based translation system on the **English-to-German** task.

| # | System | Paradigm |
|---|---|---|
| 1 | Custom Seq2Seq + Attention | Trained from scratch on Multi30k subset |
| 2 | `Helsinki-NLP/opus-mt-en-de` | Pre-trained Transformer (MarianMT), zero fine-tuning |

Translation quality is evaluated with **BLEU**, **METEOR**, **ChrF**, and **BERTScore** — covering n-gram precision, unigram alignment with synonymy, character-level overlap, and contextual semantic similarity respectively.

## 1. Environment Setup

Required libraries: `datasets` (data loading), `transformers` (MarianMT model and tokenizer), `evaluate` + `sacrebleu` (BLEU/METEOR/ChrF metrics), `bert-score` (BERTScore), `sentencepiece` (subword tokenizer backend for MarianMT), and `accelerate` (efficient inference). A global seed of **42** is applied to Python, NumPy, and PyTorch for full reproducibility.

In [ ]:
!pip -q install datasets transformers evaluate sacrebleu bert-score nltk sentencepiece accelerate

In [ ]:
import os
import re
import math
import time
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import nltk
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt_tab")

import evaluate

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Imports completed successfully.")
print("Random seed:", SEED)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")

Imports completed successfully.
Random seed: 42
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2. Dataset: Multi30k

The Multi30k dataset is used for English-to-German machine translation. It contains short image-description sentences with translations. In this experiment, English sentences are used as source inputs and German sentences are used as target outputs.

In [ ]:
dataset = load_dataset("bentrevett/multi30k")

print(dataset)

print("\nAvailable splits:")
print(dataset.keys())

print("\nColumn names:")
for split in dataset.keys():
    print(split, dataset[split].column_names)

print("\nFirst training example:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

Available splits:
dict_keys(['train', 'validation', 'test'])

Column names:
train ['en', 'de']
validation ['en', 'de']
test ['en', 'de']

First training example:
{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


To keep training time manageable, a subset of Multi30k is used. The training split is shuffled with a fixed seed, and the first 8,000 examples are selected for training. The validation split is used for monitoring, and a 100-example test subset is used for final evaluation.

In [ ]:

# Create reproducible subsets
TRAIN_SIZE = 8000
VALID_SIZE = 1000
TEST_SIZE = 100

train_data = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
valid_data = dataset["validation"].shuffle(seed=SEED).select(range(min(VALID_SIZE, len(dataset["validation"]))))
test_data = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

print("Train subset size:", len(train_data))
print("Validation subset size:", len(valid_data))
print("Test subset size:", len(test_data))

print("\nSample:")
print("EN:", train_data[0]["en"])
print("DE:", train_data[0]["de"])

Train subset size: 8000
Validation subset size: 1000
Test subset size: 100

Sample:
EN: A lady wearing green and white shorts and top is on the beach clapping her hands.
DE: Eine Dame mit grün-weißen Shorts und Oberteil ist auf dem Strand und klatscht in die Hände.


In [ ]:
# Dataset statistics

def simple_tokenize(text):
    """
    Simple tokenizer for length statistics and vocabulary construction.
    Lowercases text and separates punctuation.
    """
    text = text.lower().strip()
    text = re.sub(r"([.!?,;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

train_df = pd.DataFrame(train_data)
valid_df = pd.DataFrame(valid_data)
test_df = pd.DataFrame(test_data)

train_df["en_len"] = train_df["en"].apply(lambda x: len(simple_tokenize(x)))
train_df["de_len"] = train_df["de"].apply(lambda x: len(simple_tokenize(x)))

valid_df["en_len"] = valid_df["en"].apply(lambda x: len(simple_tokenize(x)))
valid_df["de_len"] = valid_df["de"].apply(lambda x: len(simple_tokenize(x)))

test_df["en_len"] = test_df["en"].apply(lambda x: len(simple_tokenize(x)))
test_df["de_len"] = test_df["de"].apply(lambda x: len(simple_tokenize(x)))

print("Training length statistics:")
print(train_df[["en_len", "de_len"]].describe())

print("\nValidation length statistics:")
print(valid_df[["en_len", "de_len"]].describe())

print("\nTest length statistics:")
print(test_df[["en_len", "de_len"]].describe())

Training length statistics:
            en_len       de_len
count  8000.000000  8000.000000
mean     13.071250    12.481250
std       4.109171     4.258775
min       4.000000     1.000000
25%      10.000000     9.000000
50%      12.000000    12.000000
75%      15.000000    15.000000
max      40.000000    43.000000

Validation length statistics:
            en_len       de_len
count  1000.000000  1000.000000
mean     13.065000    12.613000
std       3.876055     4.178839
min       4.000000     3.000000
25%      10.000000    10.000000
50%      12.000000    12.000000
75%      15.000000    15.000000
max      30.000000    33.000000

Test length statistics:
           en_len      de_len
count  100.000000  100.000000
mean    13.480000   12.920000
std      4.314598    4.407707
min      7.000000    6.000000
25%     10.000000   10.000000
50%     13.000000   12.000000
75%     16.000000   15.250000
max     29.000000   27.000000


## 3. Preprocessing and Vocabulary Construction

For the custom Seq2Seq model, source and target sentences are tokenized using a simple lowercase tokenizer. Separate vocabularies are built for English and German. Special tokens are added for padding, unknown words, sentence start, and sentence end.

In [ ]:
# Build vocabularies

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

def build_vocab(sentences, min_freq=2):
    """
    Build a word-level vocabulary from tokenized sentences.
    Tokens with frequency lower than min_freq are mapped to <unk>.
    """
    counter = Counter()

    for sentence in sentences:
        counter.update(simple_tokenize(sentence))

    vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)

    return vocab, counter

src_vocab, src_counter = build_vocab(train_df["en"], min_freq=1)
trg_vocab, trg_counter = build_vocab(train_df["de"], min_freq=1)

src_itos = {idx: token for token, idx in src_vocab.items()}
trg_itos = {idx: token for token, idx in trg_vocab.items()}

PAD_IDX = trg_vocab[PAD_TOKEN]
UNK_IDX = trg_vocab[UNK_TOKEN]
SOS_IDX = trg_vocab[SOS_TOKEN]
EOS_IDX = trg_vocab[EOS_TOKEN]

print("Source vocabulary size:", len(src_vocab))
print("Target vocabulary size:", len(trg_vocab))

print("\nMost common English tokens:")
print(src_counter.most_common(20))

print("\nMost common German tokens:")
print(trg_counter.most_common(20))

Source vocabulary size: 5955
Target vocabulary size: 8774

Most common English tokens:
[('a', 13622), ('.', 7640), ('in', 4043), ('the', 3041), ('on', 2221), ('man', 2129), ('and', 2115), ('is', 2066), ('of', 1895), ('with', 1698), (',', 1134), ('woman', 1069), ('two', 1058), ('are', 1039), ('people', 910), ('to', 861), ('at', 820), ('an', 788), ('wearing', 730), ('young', 619)]

Most common German tokens:
[('.', 7971), ('ein', 5159), ('einem', 3772), ('in', 3264), ('eine', 2754), ('und', 2533), (',', 2481), ('mit', 2427), ('auf', 2397), ('mann', 2157), ('einer', 1938), ('der', 1378), ('frau', 1134), ('die', 1104), ('zwei', 1066), ('einen', 965), ('an', 885), ('im', 843), ('von', 668), ('sich', 608)]


In [ ]:
# Step 3.1: Encoding and decoding helper functions

def encode_sentence(sentence, vocab, add_sos=False, add_eos=True):
    tokens = simple_tokenize(sentence)

    ids = []

    if add_sos:
        ids.append(vocab[SOS_TOKEN])

    ids.extend([vocab.get(token, vocab[UNK_TOKEN]) for token in tokens])

    if add_eos:
        ids.append(vocab[EOS_TOKEN])

    return ids

def decode_ids(ids, itos):
    tokens = []

    for idx in ids:
        token = itos.get(int(idx), UNK_TOKEN)

        if token in [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN]:
            continue

        tokens.append(token)

    return " ".join(tokens)

sample_en = train_df.iloc[0]["en"]
sample_de = train_df.iloc[0]["de"]

encoded_en = encode_sentence(sample_en, src_vocab, add_sos=False, add_eos=True)
encoded_de = encode_sentence(sample_de, trg_vocab, add_sos=True, add_eos=True)

print("Original EN:", sample_en)
print("Encoded EN:", encoded_en)
print("Decoded EN:", decode_ids(encoded_en, src_itos))

print("\nOriginal DE:", sample_de)
print("Encoded DE:", encoded_de)
print("Decoded DE:", decode_ids(encoded_de, trg_itos))

Original EN: A lady wearing green and white shorts and top is on the beach clapping her hands.
Encoded EN: [4, 5, 6, 7, 8, 9, 10, 8, 11, 12, 13, 14, 15, 16, 17, 18, 19, 3]
Decoded EN: a lady wearing green and white shorts and top is on the beach clapping her hands .

Original DE: Eine Dame mit grün-weißen Shorts und Oberteil ist auf dem Strand und klatscht in die Hände.
Encoded DE: [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 9, 15, 16, 17, 18, 19, 3]
Decoded DE: eine dame mit grün-weißen shorts und oberteil ist auf dem strand und klatscht in die hände .


## 4. PyTorch Dataset and DataLoader

The custom Seq2Seq model requires numerical token IDs instead of raw text. Each English sentence is encoded with the source vocabulary, and each German sentence is encoded with the target vocabulary. Padding is applied dynamically inside each batch so that sentences in the same batch have equal length.

In [ ]:
# Step 4: PyTorch dataset and collate function

class TranslationDataset(Dataset):
    def __init__(self, dataframe, src_vocab, trg_vocab):
        self.dataframe = dataframe.reset_index(drop=True)
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        src_text = self.dataframe.loc[idx, "en"]
        trg_text = self.dataframe.loc[idx, "de"]

        src_ids = encode_sentence(
            src_text,
            self.src_vocab,
            add_sos=False,
            add_eos=True
        )

        trg_ids = encode_sentence(
            trg_text,
            self.trg_vocab,
            add_sos=True,
            add_eos=True
        )

        return {
            "src_ids": torch.tensor(src_ids, dtype=torch.long),
            "trg_ids": torch.tensor(trg_ids, dtype=torch.long),
            "src_text": src_text,
            "trg_text": trg_text
        }


def collate_fn(batch):
    src_sequences = [item["src_ids"] for item in batch]
    trg_sequences = [item["trg_ids"] for item in batch]

    src_padded = nn.utils.rnn.pad_sequence(
        src_sequences,
        batch_first=True,
        padding_value=src_vocab[PAD_TOKEN]
    )

    trg_padded = nn.utils.rnn.pad_sequence(
        trg_sequences,
        batch_first=True,
        padding_value=trg_vocab[PAD_TOKEN]
    )

    src_texts = [item["src_text"] for item in batch]
    trg_texts = [item["trg_text"] for item in batch]

    return src_padded, trg_padded, src_texts, trg_texts

In [ ]:
# Step 4.1: Create dataloaders

BATCH_SIZE = 64

train_dataset = TranslationDataset(train_df, src_vocab, trg_vocab)
valid_dataset = TranslationDataset(valid_df, src_vocab, trg_vocab)
test_dataset = TranslationDataset(test_df, src_vocab, trg_vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

src_batch, trg_batch, src_texts, trg_texts = next(iter(train_loader))

print("Source batch shape:", src_batch.shape)
print("Target batch shape:", trg_batch.shape)

print("\nExample source text:")
print(src_texts[0])

print("\nExample target text:")
print(trg_texts[0])

print("\nExample source IDs:")
print(src_batch[0])

print("\nExample target IDs:")
print(trg_batch[0])

Source batch shape: torch.Size([64, 35])
Target batch shape: torch.Size([64, 37])

Example source text:
A group of four people congregate on a sidewalk.

Example target text:
Eine Gruppe von vier Leuten auf dem Gehweg.

Example source IDs:
tensor([  4, 360,  27, 290,  79, 708,  13,   4, 100,  19,   3,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0])

Example target IDs:
tensor([  2,   4,  95,  31, 305, 675,  12,  13, 101,  19,   3,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0])


## 5. Seq2Seq Model with Bahdanau Attention

The architecture consists of three components: a **bidirectional GRU encoder** that encodes the source sentence into a sequence of hidden states, a **Bahdanau attention module** that computes a dynamic weighted sum of encoder states at each decoding step, and a **unidirectional GRU decoder** that generates the German translation token by token conditioned on the attention context vector.

This section implements a recurrent Seq2Seq model with attention. The encoder reads the English source sentence using a bidirectional GRU. The decoder generates the German translation one token at a time. At each decoding step, the attention mechanism computes a weighted context vector over the encoder hidden states, allowing the decoder to focus on different source tokens while generating each target token.

In [ ]:
# Encoder, Attention, Decoder, and Seq2Seq model

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            emb_dim,
            padding_idx=pad_idx
        )

        self.rnn = nn.GRU(
            emb_dim,
            enc_hid_dim,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [batch_size, src_len]

        embedded = self.dropout(self.embedding(src))
        # embedded: [batch_size, src_len, emb_dim]

        outputs, hidden = self.rnn(embedded)
        # outputs: [batch_size, src_len, enc_hid_dim * 2]
        # hidden: [2, batch_size, enc_hid_dim]

        # Concatenate final forward and backward hidden states
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]

        hidden = torch.tanh(
            self.fc(torch.cat((hidden_forward, hidden_backward), dim=1))
        )
        # hidden: [batch_size, dec_hid_dim]

        return outputs, hidden


class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()

        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask):
        # hidden: [batch_size, dec_hid_dim]
        # encoder_outputs: [batch_size, src_len, enc_hid_dim * 2]
        # mask: [batch_size, src_len]

        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        # hidden: [batch_size, src_len, dec_hid_dim]

        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )
        # energy: [batch_size, src_len, dec_hid_dim]

        attention = self.v(energy).squeeze(2)
        # attention: [batch_size, src_len]

        # Mask padding positions so the decoder does not attend to <pad>.
        attention = attention.masked_fill(mask == 0, -1e10)

        return torch.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        emb_dim,
        enc_hid_dim,
        dec_hid_dim,
        dropout,
        attention,
        pad_idx
    ):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(
            output_dim,
            emb_dim,
            padding_idx=pad_idx
        )

        self.rnn = nn.GRU(
            (enc_hid_dim * 2) + emb_dim,
            dec_hid_dim,
            batch_first=True
        )

        self.fc_out = nn.Linear(
            (enc_hid_dim * 2) + dec_hid_dim + emb_dim,
            output_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs, mask):
        # input_token: [batch_size]
        # hidden: [batch_size, dec_hid_dim]

        input_token = input_token.unsqueeze(1)
        # input_token: [batch_size, 1]

        embedded = self.dropout(self.embedding(input_token))
        # embedded: [batch_size, 1, emb_dim]

        attention_weights = self.attention(hidden, encoder_outputs, mask)
        # attention_weights: [batch_size, src_len]

        attention_weights = attention_weights.unsqueeze(1)
        # attention_weights: [batch_size, 1, src_len]

        weighted_context = torch.bmm(attention_weights, encoder_outputs)
        # weighted_context: [batch_size, 1, enc_hid_dim * 2]

        rnn_input = torch.cat((embedded, weighted_context), dim=2)
        # rnn_input: [batch_size, 1, emb_dim + enc_hid_dim * 2]

        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        # output: [batch_size, 1, dec_hid_dim]
        # hidden: [1, batch_size, dec_hid_dim]

        output = output.squeeze(1)
        hidden = hidden.squeeze(0)
        embedded = embedded.squeeze(1)
        weighted_context = weighted_context.squeeze(1)

        prediction = self.fc_out(
            torch.cat((output, weighted_context, embedded), dim=1)
        )
        # prediction: [batch_size, output_dim]

        return prediction, hidden, attention_weights.squeeze(1)


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.device = device

    def create_mask(self, src):
        # mask: 1 for real tokens, 0 for padding
        return (src != self.src_pad_idx).to(self.device)

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # trg: [batch_size, trg_len]

        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            batch_size,
            trg_len,
            trg_vocab_size
        ).to(self.device)

        encoder_outputs, hidden = self.encoder(src)
        mask = self.create_mask(src)

        input_token = trg[:, 0]  # <sos>

        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(
                input_token,
                hidden,
                encoder_outputs,
                mask
            )

            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input_token = trg[:, t] if teacher_force else top1

        return outputs

In [ ]:
# Step 5.1: Initialize Seq2Seq model
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256
DROPOUT = 0.3

SRC_PAD_IDX = src_vocab[PAD_TOKEN]
TRG_PAD_IDX = trg_vocab[PAD_TOKEN]

attention = BahdanauAttention(
    enc_hid_dim=ENC_HID_DIM,
    dec_hid_dim=DEC_HID_DIM
)

encoder = Encoder(
    input_dim=INPUT_DIM,
    emb_dim=ENC_EMB_DIM,
    enc_hid_dim=ENC_HID_DIM,
    dec_hid_dim=DEC_HID_DIM,
    dropout=DROPOUT,
    pad_idx=SRC_PAD_IDX
)

decoder = Decoder(
    output_dim=OUTPUT_DIM,
    emb_dim=DEC_EMB_DIM,
    enc_hid_dim=ENC_HID_DIM,
    dec_hid_dim=DEC_HID_DIM,
    dropout=DROPOUT,
    attention=attention,
    pad_idx=TRG_PAD_IDX
)

seq2seq_model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    src_pad_idx=SRC_PAD_IDX,
    device=device
).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Seq2Seq model initialized.")
print("Trainable parameters:", count_parameters(seq2seq_model))

Seq2Seq model initialized.
Trainable parameters: 11366598


In [ ]:
# Step 5.2: Forward pass test

src_batch, trg_batch, _, _ = next(iter(train_loader))

src_batch = src_batch.to(device)
trg_batch = trg_batch.to(device)

with torch.no_grad():
    output = seq2seq_model(src_batch, trg_batch, teacher_forcing_ratio=0.5)

print("Input source batch shape:", src_batch.shape)
print("Input target batch shape:", trg_batch.shape)
print("Model output shape:", output.shape)
print("Expected output shape: [batch_size, target_length, target_vocab_size]")

Input source batch shape: torch.Size([64, 25])
Input target batch shape: torch.Size([64, 24])
Model output shape: torch.Size([64, 24, 8774])
Expected output shape: [batch_size, target_length, target_vocab_size]


## 5.1 Training the Seq2Seq Model

The Seq2Seq model is trained with the **Adam optimizer** (lr = 0.001) for up to 10 epochs using **teacher forcing** at a ratio of 0.5 — at each decoding step, the decoder receives the true previous target token with 50% probability and its own previous prediction otherwise. This scheduled exposure reduces training/inference mismatch. **Gradient clipping** (max norm = 1.0) prevents exploding gradients in the recurrent layers. Cross-entropy loss is computed over the target vocabulary, with padding positions excluded. The best model checkpoint (lowest validation loss) is saved and reloaded for inference.

In [ ]:
# Training and evaluation functions

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)


def train_epoch(model, dataloader, optimizer, criterion, clip=1.0, teacher_forcing_ratio=0.5):
    model.train()

    epoch_loss = 0

    for src, trg, _, _ in dataloader:
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(
            src,
            trg,
            teacher_forcing_ratio=teacher_forcing_ratio
        )

        # output: [batch_size, trg_len, output_dim]
        # trg: [batch_size, trg_len]

        output_dim = output.shape[-1]

        # Ignore first token because it is <sos>.
        output = output[:, 1:, :].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


def evaluate_epoch(model, dataloader, criterion):
    model.eval()

    epoch_loss = 0

    with torch.no_grad():
        for src, trg, _, _ in dataloader:
            src = src.to(device)
            trg = trg.to(device)

            output = model(
                src,
                trg,
                teacher_forcing_ratio=0.0
            )

            output_dim = output.shape[-1]

            output = output[:, 1:, :].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [ ]:
# Train Seq2Seq model


N_EPOCHS = 10
CLIP = 1.0
TEACHER_FORCING_RATIO = 0.5

best_valid_loss = float("inf")

train_losses = []
valid_losses = []

training_start_time = time.time()

for epoch in range(1, N_EPOCHS + 1):
    epoch_start_time = time.time()

    train_loss = train_epoch(
        model=seq2seq_model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        clip=CLIP,
        teacher_forcing_ratio=TEACHER_FORCING_RATIO
    )

    valid_loss = evaluate_epoch(
        model=seq2seq_model,
        dataloader=valid_loader,
        criterion=criterion
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    epoch_time = time.time() - epoch_start_time

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(seq2seq_model.state_dict(), "q4_best_seq2seq_attention.pt")

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Train PPL: {math.exp(train_loss):.2f} | "
        f"Valid PPL: {math.exp(valid_loss):.2f} | "
        f"Time: {epoch_time:.2f}s"
    )

training_time = time.time() - training_start_time

print("\nTraining completed.")
print("Best validation loss:", round(best_valid_loss, 4))
print("Total training time in seconds:", round(training_time, 2))
print("Model saved as q4_best_seq2seq_attention.pt")

Epoch 01 | Train Loss: 5.3826 | Valid Loss: 5.0272 | Train PPL: 217.60 | Valid PPL: 152.50 | Time: 4.17s
Epoch 02 | Train Loss: 4.2292 | Valid Loss: 4.5362 | Train PPL: 68.66 | Valid PPL: 93.34 | Time: 4.14s
Epoch 03 | Train Loss: 3.5458 | Valid Loss: 4.2217 | Train PPL: 34.67 | Valid PPL: 68.15 | Time: 4.13s
Epoch 04 | Train Loss: 3.0152 | Valid Loss: 4.1447 | Train PPL: 20.39 | Valid PPL: 63.10 | Time: 4.15s
Epoch 05 | Train Loss: 2.5784 | Valid Loss: 4.1130 | Train PPL: 13.18 | Valid PPL: 61.13 | Time: 4.23s
Epoch 06 | Train Loss: 2.1916 | Valid Loss: 4.1498 | Train PPL: 8.95 | Valid PPL: 63.42 | Time: 4.12s
Epoch 07 | Train Loss: 1.9569 | Valid Loss: 4.1451 | Train PPL: 7.08 | Valid PPL: 63.12 | Time: 4.11s
Epoch 08 | Train Loss: 1.7634 | Valid Loss: 4.1856 | Train PPL: 5.83 | Valid PPL: 65.73 | Time: 4.12s
Epoch 09 | Train Loss: 1.6518 | Valid Loss: 4.2524 | Train PPL: 5.22 | Valid PPL: 70.27 | Time: 4.33s
Epoch 10 | Train Loss: 1.5072 | Valid Loss: 4.2980 | Train PPL: 4.51 | Vali

After training, the best Seq2Seq checkpoint is loaded. The model then translates English sentences greedily, one token at a time. At each step, the decoder predicts the most likely next German token. The process stops when the end-of-sentence token is generated or when the maximum output length is reached.

In [ ]:
# Load best Seq2Seq model checkpoint
seq2seq_model.load_state_dict(
    torch.load("q4_best_seq2seq_attention.pt", map_location=device)
)

seq2seq_model.eval()

print("Best Seq2Seq attention model loaded.")

Best Seq2Seq attention model loaded.


## 5.2 Greedy Decoding for Seq2Seq Translation

In [ ]:
# Greedy decoding for Seq2Seq translation with repetition control

def translate_seq2seq(sentence, model, src_vocab, trg_vocab, trg_itos, max_len=None, max_token_repeat=2):
    """
    Translate an English sentence into German using greedy decoding.

    This version adds a simple repetition control mechanism:
    if the top predicted token has already appeared too many times,
    the decoder selects the next best token instead.
    """

    model.eval()

    src_ids = encode_sentence(
        sentence,
        src_vocab,
        add_sos=False,
        add_eos=True
    )

    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    # Dynamic maximum length based on source length
    if max_len is None:
        max_len = min(50, len(src_ids) + 10)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)
        mask = model.create_mask(src_tensor)

    input_token = torch.tensor([trg_vocab[SOS_TOKEN]], dtype=torch.long).to(device)

    generated_tokens = []
    token_counts = Counter()

    banned_tokens = {
        trg_vocab[PAD_TOKEN],
        trg_vocab[SOS_TOKEN],
        trg_vocab[UNK_TOKEN]
    }

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, attention_weights = model.decoder(
                input_token,
                hidden,
                encoder_outputs,
                mask
            )

        # Sort tokens by probability from highest to lowest
        sorted_token_ids = torch.argsort(output.squeeze(0), descending=True)

        predicted_token = None

        for token_id in sorted_token_ids:
            token_id = token_id.item()

            # Do not generate pad, sos, or unk
            if token_id in banned_tokens:
                continue

            # Stop if EOS is predicted
            if token_id == trg_vocab[EOS_TOKEN]:
                predicted_token = token_id
                break

            # Avoid excessive repetition
            if token_counts[token_id] < max_token_repeat:
                predicted_token = token_id
                break

        if predicted_token is None:
            break

        if predicted_token == trg_vocab[EOS_TOKEN]:
            break

        generated_tokens.append(predicted_token)
        token_counts[predicted_token] += 1

        input_token = torch.tensor([predicted_token], dtype=torch.long).to(device)

    translated_sentence = decode_ids(generated_tokens, trg_itos)

    return translated_sentence

In [ ]:
# Test Seq2Seq translation on sample examples

for idx in range(5):
    source = test_df.loc[idx, "en"]
    reference = test_df.loc[idx, "de"]
    prediction = translate_seq2seq(
        source,
        seq2seq_model,
        src_vocab,
        trg_vocab,
        trg_itos,
        max_len=50
    )

    print("=" * 80)
    print("SOURCE EN:")
    print(source)

    print("\nREFERENCE DE:")
    print(reference)

    print("\nSEQ2SEQ PREDICTION:")
    print(prediction)

SOURCE EN:
A team of cyclists rounds a bend while nearby spectators cheer and take photographs.

REFERENCE DE:
Ein Team von Radfahrern fährt um die Kurve während in Zuschauer der Nähe jubeln und Fotos machen.

SEQ2SEQ PREDICTION:
eine team , die an einem baum und und , während sie dabei .
SOURCE EN:
A boy at a gun range aims and shoots.

REFERENCE DE:
Ein Junge zielt und schießt auf einem Schießstand.

SEQ2SEQ PREDICTION:
ein junge , ein paar und und , wie er .
SOURCE EN:
A man on a scaffold in front of a house is smiling and posing for the photographer.

REFERENCE DE:
Ein Mann auf einem Gerüst vor einem Haus lächelt und posiert für den Fotografen.

SEQ2SEQ PREDICTION:
ein mann auf einem vor vor einem und und posiert für für .
SOURCE EN:
A man and a woman in white shirts are hugging each other.

REFERENCE DE:
Ein Mann und eine Frau in weißen Shirts umarmen sich.

SEQ2SEQ PREDICTION:
ein mann und eine frau in weiß gekleidet stehen einander .
SOURCE EN:
Two people are sitting fishing on 

## 6. Transformer-Based Translation Model (MarianMT)

For comparison, the pretrained **`Helsinki-NLP/opus-mt-en-de`** (MarianMT) model is applied without any fine-tuning. It was trained on large-scale parallel corpora from the OPUS collection and uses **SentencePiece subword tokenization**, which handles rare words, morphological variation, and out-of-vocabulary terms more robustly than the word-level vocabulary of the custom Seq2Seq model. Translation is performed with **beam search** (4 beams), producing higher-quality output than greedy decoding.

In [ ]:
# Load pretrained Transformer translation model

TRANSFORMER_MODEL_NAME = "Helsinki-NLP/opus-mt-en-de"

transformer_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
transformer_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSFORMER_MODEL_NAME)

transformer_model = transformer_model.to(device)
transformer_model.eval()

print("Transformer translation model loaded successfully.")
print("Model:", TRANSFORMER_MODEL_NAME)
print("Device:", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Transformer translation model loaded successfully.
Model: Helsinki-NLP/opus-mt-en-de
Device: cuda


In [40]:
# Transformer translation function

def translate_transformer(sentence, tokenizer, model, max_length=80, num_beams=4):
    """
    Translate an English sentence into German using a pretrained Transformer model.
    """

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )

    translation = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return translation

In [41]:
# Test Transformer translation on sample examples

for idx in range(5):
    source = test_df.loc[idx, "en"]
    reference = test_df.loc[idx, "de"]

    seq2seq_prediction = translate_seq2seq(
        source,
        seq2seq_model,
        src_vocab,
        trg_vocab,
        trg_itos,
        max_len=None,
        max_token_repeat=2
    )

    transformer_prediction = translate_transformer(
        source,
        transformer_tokenizer,
        transformer_model,
        max_length=80,
        num_beams=4
    )

    print("=" * 80)
    print("SOURCE EN:")
    print(source)

    print("\nREFERENCE DE:")
    print(reference)

    print("\nSEQ2SEQ + ATTENTION PREDICTION:")
    print(seq2seq_prediction)

    print("\nTRANSFORMER PREDICTION:")
    print(transformer_prediction)

SOURCE EN:
A team of cyclists rounds a bend while nearby spectators cheer and take photographs.

REFERENCE DE:
Ein Team von Radfahrern fährt um die Kurve während in Zuschauer der Nähe jubeln und Fotos machen.

SEQ2SEQ + ATTENTION PREDICTION:
eine team , die an einem baum und und , während sie dabei .

TRANSFORMER PREDICTION:
Ein Team von Radfahrern rundet eine Kurve ab, während die Zuschauer in der Nähe jubeln und fotografieren.
SOURCE EN:
A boy at a gun range aims and shoots.

REFERENCE DE:
Ein Junge zielt und schießt auf einem Schießstand.

SEQ2SEQ + ATTENTION PREDICTION:
ein junge , ein paar und und , wie er .

TRANSFORMER PREDICTION:
Ein Junge auf einem Schießstand zielt und schießt.
SOURCE EN:
A man on a scaffold in front of a house is smiling and posing for the photographer.

REFERENCE DE:
Ein Mann auf einem Gerüst vor einem Haus lächelt und posiert für den Fotografen.

SEQ2SEQ + ATTENTION PREDICTION:
ein mann auf einem vor vor einem und und posiert für für .

TRANSFORMER PREDICT

## 7. Generating Predictions for Evaluation

Both models are now used to translate the same 100 English test sentences. The generated German translations are stored together with the source sentences and reference translations. These outputs are then used for automatic evaluation.

In [42]:
# Generate translations for the test subset


from tqdm.auto import tqdm

seq2seq_predictions = []
transformer_predictions = []

translation_start_time = time.time()

for source in tqdm(test_df["en"], desc="Translating test set"):
    seq2seq_pred = translate_seq2seq(
        source,
        seq2seq_model,
        src_vocab,
        trg_vocab,
        trg_itos,
        max_len=None,
        max_token_repeat=2
    )

    transformer_pred = translate_transformer(
        source,
        transformer_tokenizer,
        transformer_model,
        max_length=80,
        num_beams=4
    )

    seq2seq_predictions.append(seq2seq_pred)
    transformer_predictions.append(transformer_pred)

translation_time = time.time() - translation_start_time

test_df["seq2seq_attention_prediction"] = seq2seq_predictions
test_df["transformer_prediction"] = transformer_predictions

print("Generated translations for", len(test_df), "test examples.")
print("Total translation time in seconds:", round(translation_time, 2))
print("Average translation time per example in seconds:", round(translation_time / len(test_df), 2))

test_df[
    [
        "en",
        "de",
        "seq2seq_attention_prediction",
        "transformer_prediction"
    ]
].head()

Translating test set:   0%|          | 0/100 [00:00<?, ?it/s]

Generated translations for 100 test examples.
Total translation time in seconds: 10.26
Average translation time per example in seconds: 0.1


,en,de,seq2seq_attention_prediction,transformer_prediction
0,A team of cyclists rounds a bend while nearby ...,Ein Team von Radfahrern fährt um die Kurve wäh...,"eine team , die an einem baum und und , währen...","Ein Team von Radfahrern rundet eine Kurve ab, ..."
1,A boy at a gun range aims and shoots.,Ein Junge zielt und schießt auf einem Schießst...,"ein junge , ein paar und und , wie er .",Ein Junge auf einem Schießstand zielt und schi...
2,A man on a scaffold in front of a house is smi...,Ein Mann auf einem Gerüst vor einem Haus läche...,ein mann auf einem vor vor einem und und posie...,Ein Mann auf einem Gerüst vor einem Haus läche...
3,A man and a woman in white shirts are hugging ...,Ein Mann und eine Frau in weißen Shirts umarme...,ein mann und eine frau in weiß gekleidet stehe...,Ein Mann und eine Frau in weißen Hemden umarme...
4,Two people are sitting fishing on striped beac...,Zwei Personen sitzen auf gestreiften Liegestüh...,zwei personen sitzen auf einem gestreiften und...,Zwei Leute sitzen auf gestreiften Strandstühle...


## 8. Evaluation Metrics

Four metrics are computed for both systems against the German reference translations:

- **BLEU** (SacreBLEU): corpus-level n-gram precision (1–4) with a brevity penalty; the standard MT benchmark metric
- **METEOR**: unigram alignment with stemming and synonym matching via WordNet; more lenient than BLEU for paraphrastic output
- **ChrF**: character n-gram F-score; particularly effective for morphologically rich languages like German where word forms vary by case, gender, and number
- **BERTScore**: token-level cosine similarity using contextual embeddings; captures semantic equivalence beyond surface overlap

In [43]:
# Load evaluation metrics

bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
chrf_metric = evaluate.load("chrf")
bertscore_metric = evaluate.load("bertscore")

references = test_df["de"].tolist()
seq2seq_preds = test_df["seq2seq_attention_prediction"].tolist()
transformer_preds = test_df["transformer_prediction"].tolist()

print("Metrics loaded successfully.")
print("Number of references:", len(references))
print("Number of Seq2Seq predictions:", len(seq2seq_preds))
print("Number of Transformer predictions:", len(transformer_preds))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Metrics loaded successfully.
Number of references: 100
Number of Seq2Seq predictions: 100
Number of Transformer predictions: 100


In [45]:
# Metric computation helper

def compute_mt_metrics(predictions, references, model_name):
    """
    Compute BLEU, METEOR, ChrF, and BERTScore for machine translation.
    """

    formatted_references = [[ref] for ref in references]

    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=formatted_references
    )

    meteor_result = meteor_metric.compute(
        predictions=predictions,
        references=references
    )

    chrf_result = chrf_metric.compute(
        predictions=predictions,
        references=formatted_references
    )

    bertscore_result = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="de"
    )

    return {
        "model": model_name,
        "bleu": bleu_result["score"],
        "meteor": meteor_result["meteor"],
        "chrf": chrf_result["score"],
        "bertscore_precision": float(np.mean(bertscore_result["precision"])),
        "bertscore_recall": float(np.mean(bertscore_result["recall"])),
        "bertscore_f1": float(np.mean(bertscore_result["f1"])),
    }

In [46]:
# Evaluate both translation systems

seq2seq_results = compute_mt_metrics(
    predictions=seq2seq_preds,
    references=references,
    model_name="Seq2Seq + Attention"
)

transformer_results = compute_mt_metrics(
    predictions=transformer_preds,
    references=references,
    model_name="Pretrained Transformer"
)

q4_results_df = pd.DataFrame([seq2seq_results, transformer_results])

q4_results_df_rounded = q4_results_df.copy()
numeric_cols = q4_results_df_rounded.columns.drop("model")
q4_results_df_rounded[numeric_cols] = q4_results_df_rounded[numeric_cols].round(4)

q4_results_df_rounded

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,bleu,meteor,chrf,bertscore_precision,bertscore_recall,bertscore_f1
0,Seq2Seq + Attention,3.2905,0.4771,35.3786,0.7499,0.7370,0.7430
1,Pretrained Transformer,36.2735,0.6752,63.7679,0.8984,0.9031,0.9006


## 9. Qualitative Analysis


A small number of examples are inspected manually to compare the model outputs. This helps show differences that are not fully captured by automatic metrics, such as fluency, rare-word handling, repetition, and whether the translation preserves the meaning of the source sentence.

In [47]:
# Qualitative examples

qualitative_indices = [0, 1, 2, 3, 4]

for idx in qualitative_indices:
    print("=" * 100)
    print(f"EXAMPLE {idx}")
    print("=" * 100)

    print("\nSOURCE EN:")
    print(test_df.loc[idx, "en"])

    print("\nREFERENCE DE:")
    print(test_df.loc[idx, "de"])

    print("\nSEQ2SEQ + ATTENTION:")
    print(test_df.loc[idx, "seq2seq_attention_prediction"])

    print("\nPRETRAINED TRANSFORMER:")
    print(test_df.loc[idx, "transformer_prediction"])

    print("\n")

EXAMPLE 0

SOURCE EN:
A team of cyclists rounds a bend while nearby spectators cheer and take photographs.

REFERENCE DE:
Ein Team von Radfahrern fährt um die Kurve während in Zuschauer der Nähe jubeln und Fotos machen.

SEQ2SEQ + ATTENTION:
eine team , die an einem baum und und , während sie dabei .

PRETRAINED TRANSFORMER:
Ein Team von Radfahrern rundet eine Kurve ab, während die Zuschauer in der Nähe jubeln und fotografieren.


EXAMPLE 1

SOURCE EN:
A boy at a gun range aims and shoots.

REFERENCE DE:
Ein Junge zielt und schießt auf einem Schießstand.

SEQ2SEQ + ATTENTION:
ein junge , ein paar und und , wie er .

PRETRAINED TRANSFORMER:
Ein Junge auf einem Schießstand zielt und schießt.


EXAMPLE 2

SOURCE EN:
A man on a scaffold in front of a house is smiling and posing for the photographer.

REFERENCE DE:
Ein Mann auf einem Gerüst vor einem Haus lächelt und posiert für den Fotografen.

SEQ2SEQ + ATTENTION:
ein mann auf einem vor vor einem und und posiert für für .

PRETRAINED TRAN

In [48]:
# Qualitative comparison table

q4_qualitative_df = test_df.loc[
    qualitative_indices,
    [
        "en",
        "de",
        "seq2seq_attention_prediction",
        "transformer_prediction"
    ]
].copy()

pd.set_option("display.max_colwidth", 1000)

q4_qualitative_df

,en,de,seq2seq_attention_prediction,transformer_prediction
0,A team of cyclists rounds a bend while nearby spectators cheer and take photographs.,Ein Team von Radfahrern fährt um die Kurve während in Zuschauer der Nähe jubeln und Fotos machen.,"eine team , die an einem baum und und , während sie dabei .","Ein Team von Radfahrern rundet eine Kurve ab, während die Zuschauer in der Nähe jubeln und fotografieren."
1,A boy at a gun range aims and shoots.,Ein Junge zielt und schießt auf einem Schießstand.,"ein junge , ein paar und und , wie er .",Ein Junge auf einem Schießstand zielt und schießt.
2,A man on a scaffold in front of a house is smiling and posing for the photographer.,Ein Mann auf einem Gerüst vor einem Haus lächelt und posiert für den Fotografen.,ein mann auf einem vor vor einem und und posiert für für .,Ein Mann auf einem Gerüst vor einem Haus lächelt und posiert für den Fotografen.
3,A man and a woman in white shirts are hugging each other.,Ein Mann und eine Frau in weißen Shirts umarmen sich.,ein mann und eine frau in weiß gekleidet stehen einander .,Ein Mann und eine Frau in weißen Hemden umarmen sich gegenseitig.
4,Two people are sitting fishing on striped beach chairs in a body of water.,Zwei Personen sitzen auf gestreiften Liegestühlen und angeln in einem Gewässer.,zwei personen sitzen auf einem gestreiften und und auf einem gewässer .,Zwei Leute sitzen auf gestreiften Strandstühlen in einem Körper aus Wasser angeln.


## 10. Results and Export

Three output files are saved:

- **`q4_machine_translation_metric_results.csv`** — BLEU, METEOR, ChrF, and BERTScore for both systems
- **`q4_generated_translations.csv`** — source sentences, reference translations, and model predictions for all 100 test examples
- **`q4_qualitative_examples.csv`** — side-by-side comparison for the 5 manually inspected examples

In [49]:
q4_results_df_rounded.to_csv("q4_machine_translation_metric_results.csv", index=False)

test_df[
    [
        "en",
        "de",
        "seq2seq_attention_prediction",
        "transformer_prediction"
    ]
].to_csv("q4_generated_translations.csv", index=False)

q4_qualitative_df.to_csv("q4_qualitative_examples.csv", index=False)

print("Saved files:")
print("- q4_machine_translation_metric_results.csv")
print("- q4_generated_translations.csv")
print("- q4_qualitative_examples.csv")

Saved files:
- q4_machine_translation_metric_results.csv
- q4_generated_translations.csv
- q4_qualitative_examples.csv


In [50]:
#Final experiment summary

print("Question 4 experiment completed.")
print("Dataset: Multi30k")
print("Task: English-to-German translation")
print("Train subset size:", len(train_df))
print("Validation subset size:", len(valid_df))
print("Test subset size:", len(test_df))
print("Random seed:", SEED)
print("Seq2Seq model: bidirectional GRU encoder + Bahdanau attention decoder")
print("Seq2Seq trainable parameters:", count_parameters(seq2seq_model))
print("Best Seq2Seq validation loss:", round(best_valid_loss, 4))
print("Seq2Seq training time in seconds:", round(training_time, 2))
print("Transformer model:", TRANSFORMER_MODEL_NAME)
print("Translation time in seconds:", round(translation_time, 2))
print("Average translation time per example:", round(translation_time / len(test_df), 2))

print("\nFinal metric results:")
display(q4_results_df_rounded)

Question 4 experiment completed.
Dataset: Multi30k
Task: English-to-German translation
Train subset size: 8000
Validation subset size: 1000
Test subset size: 100
Random seed: 42
Seq2Seq model: bidirectional GRU encoder + Bahdanau attention decoder
Seq2Seq trainable parameters: 11366598
Best Seq2Seq validation loss: 4.113
Seq2Seq training time in seconds: 41.96
Transformer model: Helsinki-NLP/opus-mt-en-de
Translation time in seconds: 10.26
Average translation time per example: 0.1

Final metric results:


,model,bleu,meteor,chrf,bertscore_precision,bertscore_recall,bertscore_f1
0,Seq2Seq + Attention,3.2905,0.4771,35.3786,0.7499,0.7370,0.7430
1,Pretrained Transformer,36.2735,0.6752,63.7679,0.8984,0.9031,0.9006
